In [1]:
class NodoPedido:
    def __init__(self, id_pedido: int, tipo: str, descripcion: str):
        self.id_pedido = id_pedido      
        self.tipo = tipo              
        self.descripcion = descripcion    
        self.siguiente = None             

class GestorPedidos:
    def __init__(self):
        self.cola_frente = None
        self.cola_fin = None
        self.pila_tope = None

    def registrar_pedido(self, id_pedido: int, tipo: str, descripcion: str) -> None:
        """
        Registra un pedido. Si es ESTÁNDAR va al final de la cola (FIFO).
        Si es RECLAMO va al tope de la pila (LIFO).
        """
        nuevo_nodo = NodoPedido(id_pedido, tipo, descripcion)
        
        if tipo == "ESTÁNDAR":
            if self.cola_frente is None:
                self.cola_frente = nuevo_nodo
                self.cola_fin = nuevo_nodo
            else:
                self.cola_fin.siguiente = nuevo_nodo
                self.cola_fin = nuevo_nodo
                
        elif tipo == "RECLAMO":
            nuevo_nodo.siguiente = self.pila_tope
            self.pila_tope = nuevo_nodo

    def despachar_siguiente(self) -> NodoPedido:
        """
        Despacha con prioridad estricta: Reclamos (Pila) primero, luego Estándar (Cola).
        Corrige el error del caso extremo solucionando el 'Puntero Fantasma'.
        """
        if self.pila_tope is not None:
            nodo_despachado = self.pila_tope
            self.pila_tope = self.pila_tope.siguiente
            nodo_despachado.siguiente = None
            return nodo_despachado
            
        elif self.cola_frente is not None:
            nodo_despachado = self.cola_frente
            self.cola_frente = self.cola_frente.siguiente
    
            if self.cola_frente is None:
                self.cola_fin = None
                
            nodo_despachado.siguiente = None
            return nodo_despachado
            
        return None

    def activar_contingencia_masiva(self) -> None:
        """
        Congela la cola, extrae uno a uno sus elementos desde el frente y los 
        apila en la pila de reclamos, invirtiendo su orden de forma natural.
        """
        while self.cola_frente is not None:
            nodo_actual = self.cola_frente
            self.cola_frente = self.cola_frente.siguiente
            
            nodo_actual.siguiente = self.pila_tope
            self.pila_tope = nodo_actual
            
        self.cola_fin = None

    def mostrar_estado(self):
        """Método auxiliar para verificar visualmente las estructuras en el Notebook"""
        print("ESTADO DEL GESTOR")
        pila = []
        actual = self.pila_tope
        while actual:
            pila.append(f"[{actual.id_pedido}: {actual.tipo}]")
            actual = actual.siguiente
        print("PILA (Tope -> Fin):", " -> ".join(pila) if pila else "Vacía")
        
        cola = []
        actual = self.cola_frente
        while actual:
            cola.append(f"[{actual.id_pedido}: {actual.tipo}]")
            actual = actual.siguiente
        print("COLA (Frente -> Fin):", " -> ".join(cola) if cola else "Vacía")

In [2]:
gestor = GestorPedidos()

gestor.registrar_pedido(101, "ESTÁNDAR", "Pizza Familiar")
gestor.registrar_pedido(102, "ESTÁNDAR", "Hamburguesa doble")
gestor.registrar_pedido(103, "ESTÁNDAR", "Sushi Roll")

gestor.registrar_pedido(501, "RECLAMO", "Pedido llegó frío")
gestor.registrar_pedido(502, "RECLAMO", "Falta bebida")

print("ESTADO INICIAL (Flujo Ordinario + Reclamos):")
gestor.mostrar_estado()

despachado = gestor.despachar_siguiente()
print(f"Se despachó de inmediato: ID {despachado.id_pedido} ({despachado.descripcion})")
gestor.mostrar_estado()

print("!!! ACTIVANDO MODO DE CONTINGENCIA MASIVA !!!")
gestor.activar_contingencia_masiva()
gestor.mostrar_estado()

print("Despachando los pedidos rescatados en orden invertido:")
while True:
    p = gestor.despachar_siguiente()
    if p is None:
        break
    print(f"Despachado: ID {p.id_pedido} - Tipo: {p.tipo}")

ESTADO INICIAL (Flujo Ordinario + Reclamos):
ESTADO DEL GESTOR
PILA (Tope -> Fin): [502: RECLAMO] -> [501: RECLAMO]
COLA (Frente -> Fin): [101: ESTÁNDAR] -> [102: ESTÁNDAR] -> [103: ESTÁNDAR]
Se despachó de inmediato: ID 502 (Falta bebida)
ESTADO DEL GESTOR
PILA (Tope -> Fin): [501: RECLAMO]
COLA (Frente -> Fin): [101: ESTÁNDAR] -> [102: ESTÁNDAR] -> [103: ESTÁNDAR]
!!! ACTIVANDO MODO DE CONTINGENCIA MASIVA !!!
ESTADO DEL GESTOR
PILA (Tope -> Fin): [103: ESTÁNDAR] -> [102: ESTÁNDAR] -> [101: ESTÁNDAR] -> [501: RECLAMO]
COLA (Frente -> Fin): Vacía
Despachando los pedidos rescatados en orden invertido:
Despachado: ID 103 - Tipo: ESTÁNDAR
Despachado: ID 102 - Tipo: ESTÁNDAR
Despachado: ID 101 - Tipo: ESTÁNDAR
Despachado: ID 501 - Tipo: RECLAMO
